In [1]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, log_loss
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from scipy.optimize import minimize

optuna.logging.set_verbosity(optuna.logging.WARNING)  # reduce noise

In [2]:
# ---- Download Data ----
df = yf.download("RELIANCE.NS", start="2011-01-01", end="2024-01-01", auto_adjust=True)
df = df.droplevel("Ticker", axis=1)

train = df[:"2020-12-31"]  # Jan 2011 - Dec 2020 (10 years)
test  = df["2021-01-01":]  # Jan 2021 - Jan 2024  (3 years)

print("Train:", train.index.min().date(), "to", train.index.max().date(), "|", len(train), "days")
print("Test: ", test.index.min().date(), "to", test.index.max().date(), "|", len(test), "days")

[*********************100%***********************]  1 of 1 completed

Train: 2011-01-03 to 2020-12-31 | 2463 days
Test:  2021-01-01 to 2023-12-29 | 741 days


In [3]:
def create_target(data, horizon=5):
    """Binary target: 1 if Close(T+horizon) > Close(T), else 0."""
    if isinstance(data.columns, pd.MultiIndex):
        data = data.droplevel("Ticker", axis=1)
    close = data[   "Close"]
    future_close = close.shift(-horizon)
    target = (future_close > close).astype(int)
    target.name = "target"
    return target

In [4]:
def build_features(data):
    """Engineer 22 features from OHLCV data."""
    if isinstance(data.columns, pd.MultiIndex):
        data = data.droplevel("Ticker", axis=1)
    close = data["Close"]
    volume = data["Volume"]
    feat = pd.DataFrame(index=data.index)

    # Price returns (momentum)
    feat["ret_1"]  = close.pct_change(1)
    feat["ret_5"]  = close.pct_change(5)
    feat["ret_10"] = close.pct_change(10)
    feat["ret_20"] = close.pct_change(20)

    # Moving average ratios (trend)
    feat["ma5_ratio"]  = close / close.rolling(5).mean()
    feat["ma10_ratio"] = close / close.rolling(10).mean()
    feat["ma20_ratio"] = close / close.rolling(20).mean()
    feat["ma50_ratio"] = close / close.rolling(50).mean()

    # Volatility
    feat["vol_5"]  = close.pct_change().rolling(5).std()
    feat["vol_20"] = close.pct_change().rolling(20).std()

    # RSI (14-day)
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss
    feat["rsi_14"] = 100 - (100 / (1 + rs))

    # Volume change
    feat["vol_change_1"] = volume.pct_change(1)
    feat["vol_change_5"] = volume.pct_change(5)

    # Rolling Z-scores
    feat["zscore_20"] = (close - close.rolling(20).mean()) / close.rolling(20).std()
    feat["zscore_50"] = (close - close.rolling(50).mean()) / close.rolling(50).std()

    # MACD
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    feat["macd"] = macd_line / close
    feat["macd_signal"] = signal_line / close
    feat["macd_hist"] = (macd_line - signal_line) / close

    # Bollinger Bands (20-day)
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_upper = bb_mid + 2 * bb_std
    bb_lower = bb_mid - 2 * bb_std
    feat["bb_upper_ratio"] = close / bb_upper
    feat["bb_lower_ratio"] = close / bb_lower
    feat["bb_bandwidth"] = (bb_upper - bb_lower) / bb_mid

    # ATR (14-day)
    high = data["High"]
    low = data["Low"]
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    feat["atr_14"] = true_range.rolling(14).mean() / close

    feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    return feat

In [5]:
# ---- Prepare data for modeling ----
X_train = build_features(train)
y_train = create_target(train, horizon=5)
X_test  = build_features(test)
y_test  = create_target(test, horizon=5)

# Drop NaN rows
train_combined = pd.concat([X_train, y_train], axis=1).dropna()
test_combined  = pd.concat([X_test, y_test], axis=1).dropna()

X_train_clean = train_combined.drop('target', axis=1)
y_train_clean = train_combined['target']
X_test_clean  = test_combined.drop('target', axis=1)
y_test_clean  = test_combined['target']

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled  = scaler.transform(X_test_clean)

print('X_train shape:', X_train_scaled.shape)
print('X_test shape: ', X_test_scaled.shape)
print('Target distribution (train):', y_train_clean.value_counts().to_dict())

X_train shape: (2410, 22)
X_test shape:  (692, 22)
Target distribution (train): {1: 1290, 0: 1120}


In [6]:
# ---- Custom Time-Series-Safe Stacking Ensemble with Scipy Optimize ----

class ScipyOptimizedEnsemble(BaseEstimator, ClassifierMixin):
    """
    Ensemble that generates out-of-fold probability predictions using 
    chronological TimeSeriesSplit, then finds optimal voting weights 
    using scipy.optimize.minimize to minimize log-loss, avoiding over-fitting 
    and preserving time-series integrity.
    """
    def __init__(self, estimators=None, n_splits=5):
        self.estimators = estimators
        self.n_splits = n_splits
        self.weights_ = None

    def fit(self, X, y):
        tscv = TimeSeriesSplit(n_splits=self.n_splits)
        val_indices = []
        meta_features = []

        y_array = y.values if hasattr(y, 'values') else np.asarray(y)
        self.classes_ = np.unique(y_array)

        # 1. Generate OOF probabilities respecting chronological order
        for train_idx, val_idx in tscv.split(X):
            fold_preds = []
            for name, est in self.estimators:
                cloned = clone(est)
                cloned.fit(X[train_idx], y_array[train_idx])
                # We need the predicted probabilities for the positive class (class 1)
                fold_preds.append(cloned.predict_proba(X[val_idx])[:, 1])
            meta_features.append(np.column_stack(fold_preds))
            val_indices.extend(val_idx)

        meta_X = np.vstack(meta_features)
        meta_y = y_array[val_indices]

        # 2. Optimize weights to minimize log-loss
        def loss_func(weights):
            # Enforce non-negativity and proper normalization
            weights_norm = np.clip(weights, 0, 1)
            sum_w = np.sum(weights_norm)
            if sum_w > 0:
                weights_norm = weights_norm / sum_w
            else:
                weights_norm = np.ones(len(self.estimators)) / len(self.estimators)
                
            pred_probs = np.dot(meta_X, weights_norm)
            # Clip bounds to prevent math domain error when calculating Log Loss
            pred_probs = np.clip(pred_probs, 1e-15, 1 - 1e-15)
            return log_loss(meta_y, pred_probs)
            
        initial_weights = np.ones(len(self.estimators)) / len(self.estimators)
        bounds = [(0, 1) for _ in self.estimators]
        constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})
        
        # Use Sequential Least Squares Programming (SLSQP) for bounded optimization
        result = minimize(loss_func, initial_weights, bounds=bounds, constraints=constraints, method='SLSQP')
        
        # Save and normalize final weights
        final_w = np.clip(result.x, 0, 1)
        self.weights_ = final_w / np.sum(final_w)
        
        # 3. Retrain base learners on the full dataset for inference
        self.estimators_ = []
        for name, est in self.estimators:
            cloned = clone(est)
            cloned.fit(X, y_array)
            self.estimators_.append((name, cloned))

        return self

    def predict_proba(self, X):
        # Predict class 1 probabilities iteratively
        preds = [est.predict_proba(X)[:, 1] for _, est in self.estimators_]
        meta_X = np.column_stack(preds)
        # Apply the optimized weights
        final_probs = np.dot(meta_X, self.weights_)
        return np.column_stack((1 - final_probs, final_probs))
        
    def predict(self, X):
        proba = self.predict_proba(X)[:, 1]
        return (proba >= 0.5).astype(int)

# ---- Hyperparameter Tuning for Scipy-Optimized Ensemble ----
# Base learners:
#   XGBoost      – primary, highest ceiling
#   RandomForest – secondary, decorrelated, robust to noise
#   LogReg       – tertiary, linear baseline, fully decorrelated
# Weights optimized via scipy SLSQP to minimize Out-Of-Fold Log Loss

def objective(trial):
    # XGBoost params
    xgb_lr   = trial.suggest_float("xgb_lr", 0.01, 0.3, log=True)
    xgb_depth = trial.suggest_int("xgb_depth", 3, 10)
    xgb_n    = trial.suggest_int("xgb_n", 100, 500)
    xgb_sub  = trial.suggest_float("xgb_sub", 0.5, 1.0)
    xgb_col  = trial.suggest_float("xgb_col", 0.5, 1.0)
    xgb_mcw  = trial.suggest_int("xgb_mcw", 1, 30)
    xgb_alpha = trial.suggest_float("xgb_alpha", 1e-4, 10.0, log=True)
    xgb_lambda = trial.suggest_float("xgb_lambda", 1e-4, 10.0, log=True)
    xgb_gamma = trial.suggest_float("xgb_gamma", 0.0, 5.0)

    # Random Forest params
    rf_depth = trial.suggest_int("rf_depth", 3, 12)
    rf_n     = trial.suggest_int("rf_n", 50, 400)

    # Logistic Regression params
    lr_c     = trial.suggest_float("lr_c", 1e-4, 10.0, log=True)

    xgb = XGBClassifier(
        learning_rate=xgb_lr, max_depth=xgb_depth, n_estimators=xgb_n,
        subsample=xgb_sub, colsample_bytree=xgb_col,
        min_child_weight=xgb_mcw, reg_alph=xgb_alpha,
        reg_lambda=xgb_lambda, gamma=xgb_gamma,
        random_state=42, eval_metric="logloss", n_jobs=-1
    )
    rf = RandomForestClassifier(
        max_depth=rf_depth, n_estimators=rf_n,
        random_state=42, n_jobs=-1
    )
    lr = LogisticRegression(C=lr_c, random_state=42, max_iter=2000)

    ensemble_clf = ScipyOptimizedEnsemble(
        estimators=[('xgb', xgb), ('rf', rf), ('lr', lr)],
        n_splits=5
    )

    # Outer chronological cross-validation
    tscv = TimeSeriesSplit(n_splits=5)
    scores = cross_val_score(ensemble_clf, X_train_scaled, y_train_clean,
                             cv=tscv, scoring="accuracy")
    return scores.mean()

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20, show_progress_bar=True)

print("Best trial:")
print("  Accuracy:", round(study.best_value, 4))
print("  Params:")
for k, v in study.best_params.items():
    print(f"    {k}: {v}")


  0%|          | 0/20 [00:00<?, ?it/s]

c:\Users\Tejas\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:48:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "reg_alph" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Tejas\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:48:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "reg_alph" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Tejas\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:48:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "reg_alph" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Tejas\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:48:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "reg_alph" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users

Best trial:
  Accuracy: 0.5501
  Params:
    xgb_lr: 0.010724311014441375
    xgb_depth: 8
    xgb_n: 299
    xgb_sub: 0.968324376628699
    xgb_col: 0.8331694724476859
    xgb_mcw: 9
    xgb_alpha: 6.292942010319485
    xgb_lambda: 0.00011365065005058419
    xgb_gamma: 1.9886619309571234
    rf_depth: 6
    rf_n: 257
    lr_c: 0.00011053902847282346


In [8]:
# ---- Train Best Scipy-Optimized Ensemble & Evaluate ----
bp = study.best_params

xgb_best = XGBClassifier(
    learning_rate=bp["xgb_lr"], max_depth=bp["xgb_depth"],
    n_estimators=bp["xgb_n"], subsample=bp["xgb_sub"],
    colsample_bytree=bp["xgb_col"], min_child_weight=bp["xgb_mcw"],
    reg_alpha=bp["xgb_alpha"], reg_lambda=bp["xgb_lambda"],
    gamma=bp["xgb_gamma"],
    random_state=42, eval_metric="logloss", n_jobs=-1
)
rf_best = RandomForestClassifier(
    max_depth=bp["rf_depth"], n_estimators=bp["rf_n"],
    random_state=42, n_jobs=-1
)
lr_best = LogisticRegression(C=bp["lr_c"], random_state=42, max_iter=2000)

best_model = ScipyOptimizedEnsemble(
    estimators=[('xgb', xgb_best), ('rf', rf_best), ('lr', lr_best)],
    n_splits=5
)

best_model.fit(X_train_scaled, y_train_clean)

y_train_pred_best = best_model.predict(X_train_scaled)
y_test_pred_best  = best_model.predict(X_test_scaled)

print("=== TUNED SCIPY-OPTIMIZED ENSEMBLE (XGB + RF + LR) ===")
# Show derived weights for each model
print("Optimized Weights:")
for name, w in zip(['XGBoost', 'Random Forest', 'Logistic Regression'], best_model.weights_):
    print(f"  {name}: {w:.4f}")

print("\n--- Train Set ---")
print("Accuracy:", round(accuracy_score(y_train_clean, y_train_pred_best), 4))
print(classification_report(y_train_clean, y_train_pred_best,
                            target_names=["Down/Flat", "Up"]))

print("--- Test Set ---")
print("Accuracy:", round(accuracy_score(y_test_clean, y_test_pred_best), 4))
print(classification_report(y_test_clean, y_test_pred_best,
                            target_names=["Down/Flat", "Up"]))
print("Confusion Matrix (Test):")
print(confusion_matrix(y_test_clean, y_test_pred_best))

# Set model for downstream trading simulation
model = best_model


=== TUNED SCIPY-OPTIMIZED ENSEMBLE (XGB + RF + LR) ===
Optimized Weights:
  XGBoost: 0.0740
  Random Forest: 0.2457
  Logistic Regression: 0.6804

--- Train Set ---
Accuracy: 0.6378
              precision    recall  f1-score   support

   Down/Flat       0.93      0.24      0.38      1120
          Up       0.60      0.98      0.74      1290

    accuracy                           0.64      2410
   macro avg       0.76      0.61      0.56      2410
weighted avg       0.75      0.64      0.57      2410

--- Test Set ---
Accuracy: 0.5477
              precision    recall  f1-score   support

   Down/Flat       0.57      0.13      0.22       324
          Up       0.54      0.91      0.68       368

    accuracy                           0.55       692
   macro avg       0.56      0.52      0.45       692
weighted avg       0.56      0.55      0.46       692

Confusion Matrix (Test):
[[ 43 281]
 [ 32 336]]


In [9]:
# ---- Trading Simulation (Long & Short) ----
def run_trading_simulation(test_df, model, scaler, initial_capital=1_000_000):
    """Simulate trading: Long on signal=1, Short on signal=0."""
    X_test = build_features(test_df)
    X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
    valid_idx = X_test.dropna().index
    X_test_clean = X_test.loc[valid_idx]
    X_test_scaled = scaler.transform(X_test_clean)
    predictions = pd.Series(model.predict(X_test_scaled), index=valid_idx)

    close = test_df["Close"]
    open_price = test_df["Open"]
    capital = initial_capital
    trades = []
    i = 0
    valid_dates = valid_idx.tolist()

    while i < len(valid_dates):
        signal_date = valid_dates[i]
        signal = predictions[signal_date]
        signal_pos = test_df.index.get_loc(signal_date)
        entry_pos = signal_pos + 1
        exit_pos  = signal_pos + 5
        if exit_pos >= len(test_df):
            break
        entry_date = test_df.index[entry_pos]
        exit_date  = test_df.index[exit_pos]
        entry_price = open_price.iloc[entry_pos]
        exit_price  = close.iloc[exit_pos]

        if signal == 1:
            trade_return = (exit_price / entry_price) - 1
            direction = "LONG"
        else:
            trade_return = (entry_price / exit_price) - 1
            direction = "SHORT"

        new_capital = capital * (1 + trade_return)
        trades.append({
            "signal_date": signal_date, "direction": direction,
            "entry_date": entry_date, "exit_date": exit_date,
            "entry_price": round(entry_price, 2),
            "exit_price": round(exit_price, 2),
            "trade_return": round(trade_return, 4),
            "capital_before": round(capital, 2),
            "capital_after": round(new_capital, 2),
        })
        capital = new_capital
        i += 1
        while i < len(valid_dates) and valid_dates[i] <= exit_date:
            i += 1

    return pd.DataFrame(trades), capital


def compute_financial_metrics(trades_df, initial_capital, risk_free_rate=0.05):
    """Compute Total Return and annualized Sharpe Ratio."""
    final_capital = trades_df["capital_after"].iloc[-1]
    total_return = (final_capital / initial_capital) - 1
    trade_returns = trades_df["trade_return"]
    trades_per_year = 250 / 5
    rf_per_trade = risk_free_rate / trades_per_year
    sharpe = (trade_returns.mean() - rf_per_trade) / trade_returns.std() * np.sqrt(trades_per_year)
    return total_return, sharpe, final_capital


# ---- Run Simulation ----
INITIAL_CAPITAL = 1_000_000
trades_df, final_capital = run_trading_simulation(test, model, scaler, INITIAL_CAPITAL)
total_return, sharpe, _ = compute_financial_metrics(trades_df, INITIAL_CAPITAL)

print("Initial Capital:", f"Rs.{INITIAL_CAPITAL:,.2f}")
print("Final Capital:  ", f"Rs.{final_capital:,.2f}")
print("Total Return:   ", f"{total_return * 100:.2f}%")
print("Sharpe Ratio:   ", round(sharpe, 4))
print("Total Trades:   ", len(trades_df))
long_count = (trades_df["direction"]=="LONG").sum()
short_count = (trades_df["direction"]=="SHORT").sum()
print("Long / Short:   ", long_count, "/", short_count)
win_rate = (trades_df["trade_return"] > 0).mean() * 100
print("Win Rate:       ", f"{win_rate:.1f}%")
print("\nFirst 10 trades:")
trades_df.head(10)

Initial Capital: Rs.1,000,000.00
Final Capital:   Rs.1,142,078.97
Total Return:    14.21%
Sharpe Ratio:    0.1481
Total Trades:    115
Long / Short:    105 / 10
Win Rate:        49.6%

First 10 trades:


,signal_date,direction,entry_date,exit_date,entry_price,exit_price,trade_return,capital_before,capital_after
0,2021-03-15,LONG,2021-03-16,2021-03-22,958.55,935.16,-0.0244,1000000.00,975591.28
1,2021-03-23,LONG,2021-03-24,2021-03-31,943.02,908.27,-0.0369,975591.28,939635.54
2,2021-04-01,LONG,2021-04-05,2021-04-09,918.18,898.72,-0.0212,939635.54,919728.59
3,2021-04-12,LONG,2021-04-13,2021-04-20,872.40,862.04,-0.0119,919728.59,908805.74
4,2021-04-22,LONG,2021-04-23,2021-04-29,864.24,917.77,0.0619,908805.74,965093.50
5,2021-04-30,LONG,2021-05-03,2021-05-07,891.45,875.92,-0.0174,965093.50,948280.49
6,2021-05-10,LONG,2021-05-11,2021-05-18,868.32,901.40,0.0381,948280.49,984404.20
7,2021-05-19,SHORT,2021-05-20,2021-05-26,905.91,893.28,0.0141,984404.20,998320.39
8,2021-05-27,LONG,2021-05-28,2021-06-03,902.33,1001.92,0.1104,998320.39,1108511.89
9,2021-06-04,LONG,2021-06-07,2021-06-11,998.46,1007.01,0.0086,1108511.89,1118008.94
